# バレーボール動画 AI分析 - Step 2〜3: 全区間分析 + 検証

このノートブックは、Gemini API を使用して動画と選手写真からラリー分析を行う実装です。

## Step 2〜3 の範囲
- ffmpeg 720p H.264 faststart 再エンコード
- 3分区間 + 20秒オーバーラップ分割
- 各区間を Gemini API に送信
- JSON スキーマ検証
- 絶対秒変換
- 重複排除

## まだ実装しないもの
- シート書き込み
- GAS 実装
- UI

## セル1: ライブラリインストール

In [ ]:
!pip install google-genai gdown -q

## セル2: インポート・設定

In [ ]:
import os
import json
from pathlib import Path

import google.genai as genai

try:
    from google.colab import auth, userdata
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# ローカル実行用の秘密ファイル
LOCAL_SECRET_FILE = 'analysis/.local_secrets.json'


def load_local_secrets(file_path):
    """ローカルの秘密ファイルを読み込む"""
    candidates = [
        Path(file_path),
        Path('.local_secrets.json'),
        Path('analysis/.local_secrets.json'),
    ]

    for p in candidates:
        if p.exists():
            try:
                data = json.loads(p.read_text(encoding='utf-8'))
                if isinstance(data, dict):
                    print(f'ローカル秘密ファイルを読み込みました: {p}')
                    return data
            except Exception:
                pass

    return {}


def get_secret(key_name):
    """シークレット値を取得（Colab Secrets > 環境変数 > ローカルファイル）"""
    if IN_COLAB:
        try:
            value = userdata.get(key_name)
            if value is not None:
                return str(value).strip()
        except Exception:
            pass

    env_value = str(os.getenv(key_name, '')).strip()
    if env_value:
        return env_value

    file_value = LOCAL_SECRETS.get(key_name, '')
    return str(file_value).strip()


LOCAL_SECRETS = load_local_secrets(LOCAL_SECRET_FILE)

# Gemini API Key
GEMINI_API_KEY = get_secret('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    raise ValueError(
        'GEMINI_API_KEY が設定されていません。\n'
        'Colab の場合は Secrets に GEMINI_API_KEY を登録してください。\n'
        'ローカルの場合は .local_secrets.json に {"GEMINI_API_KEY": "your-key"} を設定してください。'
    )

print('設定読み込み完了')

## セル3: Google Drive マウント

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive マウント完了')
else:
    print('ローカル実行モード: Drive マウントはスキップされます')

## セル4: Gemini クライアント初期化

In [ ]:
# Gemini クライアント初期化
client = genai.Client(api_key=GEMINI_API_KEY)

# モデル設定（設計書セクション4.4に準拠）
MODEL_NAME = "gemini-2.5-flash"
print(f'Gemini クライアント初期化完了: {MODEL_NAME}')

## セル5: Google Drive から動画をダウンロード

In [ ]:
import gdown

def download_video_from_drive(file_id, output_path):
    """Google Drive の共有リンクから動画をダウンロード（gdown使用）"""
    print(f'Google Drive から動画をダウンロード: {file_id}')
    
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, output_path, quiet=False)
    
    file_size = Path(output_path).stat().st_size
    print(f'ダウンロード完了: {output_path} ({file_size / 1024 / 1024:.2f} MB)')
    return output_path


# サンプル動画のファイルID
DRIVE_FILE_ID = "1iz8llxRGksr-jYpnwcy0FPvKeRAVjwQM"

# ダウンロード先パス
if IN_COLAB:
    DOWNLOAD_DIR = "/content"
else:
    DOWNLOAD_DIR = "analysis"

DOWNLOAD_PATH = f"{DOWNLOAD_DIR}/sample_video.mp4"

# 動画をダウンロード
download_video_from_drive(DRIVE_FILE_ID, DOWNLOAD_PATH)

# ============================================
# 動画処理設定
# ============================================

# 動画ファイル（Google Drive からダウンロードしたものを使用）
FULL_VIDEO_PATH = DOWNLOAD_PATH

# ffmpeg 720p再エンコード
if IN_COLAB:
    ENCODED_VIDEO_PATH = "/content/video_720p.mp4"
    TEMP_DIR = "/content"
else:
    ENCODED_VIDEO_PATH = "analysis/video_720p.mp4"
    TEMP_DIR = "analysis"

# ffmpeg 720p H.264 faststart 再エンコード
print(f'720p再エンコード開始: {FULL_VIDEO_PATH}')
import subprocess
cmd = [
    'ffmpeg', '-i', FULL_VIDEO_PATH,
    '-vf', 'scale=-2:720',
    '-c:v', 'libx264',
    '-movflags', '+faststart',
    '-c:a', 'aac',
    '-y',  # 上書き許可
    ENCODED_VIDEO_PATH
]
subprocess.run(cmd, check=True, capture_output=True)

from pathlib import Path
encoded_size = Path(ENCODED_VIDEO_PATH).stat().st_size
print(f'720p再エンコード完了: {ENCODED_VIDEO_PATH} ({encoded_size / 1024 / 1024:.2f} MB)')

# 動画長を取得
cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', ENCODED_VIDEO_PATH]
result = subprocess.run(cmd, capture_output=True, text=True, check=True)
video_duration = float(result.stdout.strip())
print(f'動画長: {video_duration:.1f}秒')

# 区間設定
SEGMENT_DURATION = 180  # 3分
OVERLAP_SEC = 20

# 区間分割
import math
num_segments = math.ceil((video_duration - SEGMENT_DURATION) / (SEGMENT_DURATION - OVERLAP_SEC)) + 1 if video_duration > SEGMENT_DURATION else 1
print(f'分割区間数: {num_segments}')

segments = []
for i in range(num_segments):
    start = i * (SEGMENT_DURATION - OVERLAP_SEC)
    end = min(start + SEGMENT_DURATION, video_duration)
    if start >= video_duration:
        break
    segment_path = f"{TEMP_DIR}/segment_{i:02d}.mp4"
    
    print(f'区間{i}: {start:.1f}s 〜 {end:.1f}s')
    cmd = [
        'ffmpeg', '-ss', str(start), '-to', str(end),
        '-i', ENCODED_VIDEO_PATH,
        '-c', 'copy',  # 再エンコード済みなのでcopyで高速
        '-y',
        segment_path
    ]
    subprocess.run(cmd, check=True, capture_output=True)
    
    segments.append({
        'index': i,
        'start_sec': start,
        'end_sec': end,
        'path': segment_path
    })
    segment_size = Path(segment_path).stat().st_size
    print(f'  → {segment_path} ({segment_size / 1024 / 1024:.2f} MB)')

print(f'\n合計区間数: {len(segments)}')

# 選手写真ディレクトリ
if IN_COLAB:
    PLAYERS_DIR = "/content/players"
else:
    PLAYERS_DIR = "analysis/players"

# 選手リスト
PLAYER_NAMES = []

# 試合情報
MATCH_INFO = {
    "opponent": "レジン",
    "set": 1,
    "serve_team": "レジン",
    "rotation": 1
}

print(f'選手写真ディレクトリ: {PLAYERS_DIR}')
print(f'選手リスト: {PLAYER_NAMES}')

In [ ]:
## セル10: 全区間分析実行

import time

def call_gemini_with_video_and_photos(video_file, player_photos, system_prompt, user_prompt, max_retries=3):
    """Gemini API を呼び出し（動画+選手写真+プロンプト）。リトライ付き。"""
    for attempt in range(max_retries):
        try:
            print(f'Gemini API 呼び出し開始... (試行 {attempt + 1}/{max_retries})')
            
            # コンテンツを構築
            contents = [user_prompt]
            
            # 選手写真を追加
            for photo in player_photos:
                contents.append({
                    'inline_data': {
                        'mime_type': photo['mime_type'],
                        'data': photo['data']
                    }
                })
            
            # 動画を追加
            contents.append(video_file)
            
            # API 呼び出し（システムプロンプトを含める）
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config={
                    "response_mime_type": "application/json",
                    "max_output_tokens": 65536,  # 設計書8192だが切り詰め対策で65536
                    "temperature": 0.2,  # 設計書通り（Step 2-3で問題があれば0.5へ）
                    "system_instruction": system_prompt
                }
            )
            
            print('API 呼び出し完了')
            return response
            
        except Exception as e:
            print(f'API 呼び出しエラー: {e}')
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # 指数バックオフ
                print(f'{wait_time}秒待機してリトライ...')
                time.sleep(wait_time)
            else:
                raise


def analyze_all_segments(segments, player_photos, system_prompt, match_info):
    """全区間を分析"""
    all_rallies = []
    skipped_segments = 0
    
    for seg in segments:
        print(f'\n=== 区間 {seg["index"]} ({seg["start_sec"]:.1f}s 〜 {seg["end_sec"]:.1f}s) ===')
        
        # 動画アップロード
        try:
            video_file = upload_video_to_gemini(seg['path'])
        except Exception as e:
            print(f'区間{seg["index"]} アップロード失敗: {e}')
            skipped_segments += 1
            continue
        
        # ユーザープロンプト構築
        user_prompt = build_user_prompt(match_info, seg['start_sec'], seg['end_sec'])
        
        # API 呼び出し
        try:
            response = call_gemini_with_video_and_photos(video_file, player_photos, system_prompt, user_prompt)
        except Exception as e:
            print(f'区間{seg["index"]} API 呼び出し失敗: {e}')
            try:
                delete_video_file(video_file)
            except:
                pass
            skipped_segments += 1
            continue
        
        # JSON パース・検証・正規化
        try:
            data = json.loads(response.text)
            if 'rallies' not in data:
                print(f'区間{seg["index"]} ralliesキーなし')
                skipped_segments += 1
                try:
                    delete_video_file(video_file)
                except:
                    pass
                continue
            
            rallies = data['rallies']
            print(f'区間{seg["index"]}: {len(rallies)}ラリー検出')
            
            for rally in rallies:
                # スキーマ検証・正規化
                validated, errors = validate_and_normalize_rally(rally)
                if validated is None:
                    print(f'  ラリー{rally.get("rally_number", "?")} 区間エラー: {errors}')
                    continue
                
                if errors:
                    print(f'  ラリー{rally.get("rally_number", "?")}: {len(errors)}件の警告')
                
                # 絶対秒変換
                validated = convert_to_absolute_seconds(validated, seg['start_sec'])
                validated['segment_index'] = seg['index']
                
                all_rallies.append(validated)
            
        except json.JSONDecodeError as e:
            print(f'区間{seg["index"]} JSON パースエラー: {e}')
            skipped_segments += 1
        
        # 動画削除
        try:
            delete_video_file(video_file)
        except Exception as e:
            print(f'動画削除失敗: {e}')
        
        # 5秒ウェイト
        time.sleep(5)
    
    if skipped_segments > 0:
        print(f'\n警告: {skipped_segments}区間スキップ')
    
    return all_rallies


# 全区間分析実行
print(f'全区間分析開始: {len(segments)}区間')
all_rallies = analyze_all_segments(segments, player_photos, SYSTEM_PROMPT, MATCH_INFO)
print(f'\n全区間分析完了: {len(all_rallies)}ラリー検出')

In [ ]:
## セル11: 重複排除

# 重複排除実行
print('重複排除開始...')
deduped_rallies = remove_duplicates(all_rallies, threshold_sec=5.0)
print(f'重複排除完了: {len(all_rallies)} → {len(deduped_rallies)}ラリー')

In [ ]:
## セル12: 結果表示

print('=== Step 2〜3 完了 ===')
print(f'重複排除後ラリー数: {len(deduped_rallies)}')

for i, rally in enumerate(deduped_rallies, 1):
    print(f"\n--- ラリー {i} ---")
    print(f"  絶対タイムスタンプ: {rally['absolute_start_sec']:.1f}s 〜 {rally['absolute_end_sec']:.1f}s")
    print(f"  得点チーム: {rally['point_team']}")
    print(f"  決定チーム: {rally['deciding_team']}")
    print(f"  プレー種別: {rally['play_type']}")
    print(f"  確信度: {rally['confidence']:.2f}")

In [ ]:
## セル13: 一時ファイル削除

print('一時ファイル削除開始...')

# 720p再エンコード動画
try:
    Path(ENCODED_VIDEO_PATH).unlink()
    print(f'削除: {ENCODED_VIDEO_PATH}')
except Exception as e:
    print(f'削除失敗: {ENCODED_VIDEO_PATH} - {e}')

# 区間動画
for seg in segments:
    try:
        Path(seg['path']).unlink()
        print(f'削除: {seg["path"]}')
    except Exception as e:
        print(f'削除失敗: {seg["path"]} - {e}')

print('一時ファイル削除完了')